# Module 6: MLOps avec MLflow

Objectif: Suivre les expérimentations du modèle multimodal avec **MLflow**, enregistrer les métriques et sauvegarder le modèle final.

In [1]:
import mlflow
import mlflow.sklearn
from pathlib import Path
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

DATA_DIR = Path("..") / "data" / "processed"
train_df = pd.read_csv(DATA_DIR / "train_clean.csv")
test_df = pd.read_csv(DATA_DIR / "test_clean.csv")

target_col = "Categorie"
text_col = "Rapport_Collecte"

train_df[text_col] = train_df[text_col].fillna("")
test_df[text_col] = test_df[text_col].fillna("")

num_cols = [c for c in train_df.select_dtypes(include=["number"]).columns if c not in ["Prix_Revente"]]

X_train, y_train = train_df[[text_col] + num_cols], train_df[target_col]
X_test, y_test = test_df[[text_col] + num_cols], test_df[target_col]

In [2]:
# Configuration de MLflow
project_root = Path.cwd().resolve()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent
db_path = project_root / "mlflow.db"

mlflow.set_tracking_uri(f"sqlite:///{db_path.as_posix()}")
mlflow.set_experiment("Waste_Classification_Multimodal")

# Définition des hyperparamètres à tester (5 runs)
runs_params = [
    {"n_estimators": 50, "max_depth": 10},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 50, "max_depth": 20},
    {"n_estimators": 100, "max_depth": 20},
    {"n_estimators": 150, "max_depth": 30},
]

for idx, params in enumerate(runs_params):
    with mlflow.start_run(run_name=f"Run_{idx+1}"):
        
        # Pipeline
        preprocessor = ColumnTransformer([
            ("text", TfidfVectorizer(max_features=1000, ngram_range=(1, 2)), text_col),
            ("num", "passthrough", num_cols)
        ])
        
        clf = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            random_state=42
        )
        
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("clf", clf)
        ])
        
        # Entraînement
        model.fit(X_train, y_train)
        
        # Évaluation
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        
        # Logging MLflow
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.sklearn.log_model(model, "model")
        
        print(f"Run {idx+1} | Params: {params} | Accuracy: {acc:.4f}")

# Exporter le modèle pour FastAPI
joblib.dump(model, "multimodal_model.pkl")
print("Modèle sauvegardé sous 'multimodal_model.pkl' pour FastAPI.")

2026/05/22 01:38:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/22 01:38:45 INFO mlflow.store.db.utils: Updating database tables
2026/05/22 01:38:47 INFO mlflow.tracking.fluent: Experiment with name 'Waste_Classification_Multimodal' does not exist. Creating a new experiment.
2026/05/22 01:38:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 01:38:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 1 | Params: {'n_estimators': 50, 'max_depth': 10} | Accuracy: 1.0000


2026/05/22 01:38:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 01:38:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 2 | Params: {'n_estimators': 100, 'max_depth': 10} | Accuracy: 1.0000


2026/05/22 01:39:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 01:39:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 3 | Params: {'n_estimators': 50, 'max_depth': 20} | Accuracy: 1.0000


2026/05/22 01:39:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 01:39:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 4 | Params: {'n_estimators': 100, 'max_depth': 20} | Accuracy: 1.0000


2026/05/22 01:39:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/22 01:39:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 5 | Params: {'n_estimators': 150, 'max_depth': 30} | Accuracy: 1.0000
Modèle sauvegardé sous 'multimodal_model.pkl' pour FastAPI.
